In [1]:
import pandas as pd
dataset=pd.read_json('/home/hammadali08/Personal/FYP Datasets/News_Category_Dataset_v3.json',lines=True)
dataset

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28


In [2]:
dataset.shape

(209527, 6)

In [3]:
from nltk.stem import WordNetLemmatizer
import nltk
import re
import numpy as np
import gensim
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()

In [4]:
corpus = []
for i in range(0, len(dataset)):
    review = re.sub('[^a-zA-Z0-9]', ' ', dataset['headline'][i])
    review = review.lower()
    review = review.split()

    review = [lemmatizer.lemmatize(word) for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

In [5]:
corpus1 = []
for i in range(0, len(dataset)):
    review1 = re.sub('[^a-zA-Z0-9]', ' ', dataset['short_description'][i])
    review1 = review1.lower()
    review1 = review1.split()

    review1 = [lemmatizer.lemmatize(word) for word in review1 if word not in set(stopwords.words('english'))]
    review1 = ' '.join(review1)
    corpus1.append(review1)

In [6]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [7]:
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [8]:
## Lets train Word2vec from scratch
model=gensim.models.Word2Vec(words,window=5,min_count=2)

In [9]:
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)

    return np.mean([model.wv[word]for word in doc if word in model.wv.index_to_key],axis=0)
                           #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [10]:
from tqdm import tqdm
#apply for the entire sentences
import numpy as np
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))
X

 10%|█         | 21875/209460 [00:15<02:01, 1543.22it/s]/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 209460/209460 [02:20<00:00, 1495.28it/s]


[array([ 0.2615968 , -0.29432377,  0.29975936,  0.16570602,  0.04208194,
        -0.50755024, -0.13156167,  0.09701738,  0.02768381, -0.08113565,
         0.17035986, -0.18583596, -0.06428543,  0.26170766, -0.02074121,
        -0.43076053, -0.3075474 , -0.36470565, -0.20074366, -0.24064305,
        -0.14399853,  0.2941125 ,  0.46437025,  0.1596613 ,  0.22372785,
        -0.23594946, -0.03190968, -0.15166515, -0.12094206, -0.03807436,
        -0.15980078,  0.02932621, -0.0704776 , -0.18554498, -0.10989869,
         0.1892939 , -0.28114465, -0.18032221, -0.2144396 , -0.44771752,
        -0.12647757, -0.32916632, -0.46219888,  0.21620259,  0.31420946,
         0.44032955, -0.12724133, -0.17956153, -0.01486967,  0.08195655,
         0.04914109, -0.07312907, -0.485216  , -0.49477196,  0.36871016,
         0.09934946,  0.1210312 ,  0.09032599, -0.1642619 ,  0.26313537,
         0.03854319, -0.17007828, -0.13955334, -0.08581224, -0.11508633,
         0.34138525, -0.06209705,  0.038749  , -0.4

In [11]:
words_output=[]
for sent in corpus1:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words_output.append(simple_preprocess(sent))

In [12]:
model_output=gensim.models.Word2Vec(words_output,window=5,min_count=2)

In [13]:
def avg_word2vec_output(doc_output):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)

    return np.mean([model_output.wv[word]for word in doc_output if word in model_output.wv.index_to_key],axis=0)
                           #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [28]:
Y=[]
for i in tqdm(range(len(words_output))):
    Y.append(avg_word2vec_output(words_output[i]))
Y

  0%|          | 477/189348 [00:01<06:25, 489.69it/s]/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 189348/189348 [04:18<00:00, 733.67it/s]


[array([ 0.13625193,  0.33279282,  0.21530981,  0.38068554, -0.17049785,
        -0.18066415,  0.36117893,  0.59731007, -0.28849372, -0.46810585,
        -0.07121012, -0.65051645,  0.08233866,  0.0152651 , -0.23576294,
         0.11533294, -0.00188182, -0.18355322, -0.16102147, -0.14218982,
         0.362162  ,  0.16669755,  0.25244796,  0.0962501 , -0.39121434,
         0.56010413, -0.38485414, -0.13040516,  0.15796146,  0.05884125,
         0.7102918 , -0.02890841,  0.77532023,  0.04292794,  0.16967115,
         0.49397406, -0.21624738, -0.09962376, -0.8825923 , -0.1155386 ,
        -0.33390078,  0.20660032,  0.00456947, -0.02188637,  0.5891083 ,
        -0.07574296,  0.16442065, -0.6546737 , -0.28812277, -0.18873198,
        -0.06844357, -0.4308667 , -0.1767641 ,  0.1321414 , -0.1408691 ,
        -0.31261724, -0.04200876,  0.1325799 , -0.07640685,  0.16943102,
         0.0203636 , -0.06165433,  0.04931397,  0.10414071, -0.8266678 ,
        -0.6432529 ,  0.38606787,  0.36789638, -0.1

In [32]:
df_list = []
for i in range(len(X)):
    df_list.append(pd.DataFrame(X[i].reshape(1, -1)))

df = pd.concat(df_list, ignore_index=True)

/tmp/ipykernel_5177/1423230961.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(df_list, ignore_index=True)


In [39]:
df.dropna(inplace=True)
df.shape

(209347, 100)

In [41]:
df_list_ = []
for i in range(len(Y)):
    df_list_.append(pd.DataFrame(Y[i].reshape(1, -1)))

df_ = pd.concat(df_list_, ignore_index=True)

/tmp/ipykernel_5177/2092782438.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_ = pd.concat(df_list_, ignore_index=True)


In [25]:
df_=Y

In [26]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2)

ValueError: Found input variables with inconsistent numbers of samples: [209460, 189348]